In [1]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict


In [3]:
load_dotenv()
model=ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
class CricketState(TypedDict):
    runs:int
    four:int
    six:int
    balls:int
    strike_rate:float
    boundary_p:float
    bpb:int
    summary:str
    

In [22]:
def calc_sr(state:CricketState)->CricketState:
    runs=state['runs']
    balls=state['balls']
    sr=(runs/balls)*10
    return {"strike_rate":sr}
def calc_boundary_p(state:CricketState)->CricketState:
    fours=state['four']
    sixes=state['six']
    runs=state['runs']
    boundary_p=(((fours*4)+(sixes*6))/runs)*100
    return {"boundary_p":boundary_p}

def calc_bpb(state:CricketState)->CricketState:
    fours=state['four']
    sixes=state['six']
    balls=state['balls']
    bpb=balls/(sixes+fours)
    return {"bpb":bpb}
def get_summary(state:CricketState)->CricketState:
    summary=f"""
    SR:{state['strike_rate']}
    BPB:{state['bpb']}
    Boundary_percent:{state['boundary_p']}
    """
    state['summary']=summary
    return state 

In [23]:
state=StateGraph(CricketState)

# nodes
state.add_node("calc_sr",calc_sr)
state.add_node("calc_boundary_p",calc_boundary_p)
state.add_node("calc_bpb",calc_bpb)
state.add_node("get_summary",get_summary)

# Edges
state.add_edge(START,"calc_sr")
state.add_edge(START,"calc_boundary_p")
state.add_edge(START,"calc_bpb")

state.add_edge("calc_sr","get_summary")
state.add_edge("calc_bpb","get_summary")
state.add_edge("calc_boundary_p","get_summary")

state.add_edge("get_summary",END)


In [24]:
workflow=state.compile()

In [25]:
initial_state={"runs":100,
               "balls":50,
               "four":6,
               "six":4}
workflow.invoke(initial_state)

{'runs': 100,
 'four': 6,
 'six': 4,
 'balls': 50,
 'strike_rate': 20.0,
 'boundary_p': 48.0,
 'bpb': 5.0,
 'summary': '\n    SR:20.0\n    BPB:5.0\n    Boundary_percent:48.0\n    '}